# A5' — `C.U40k.rsmiles10.e2 → Ot`: the USPTO-trained model on the ORD test

Evaluation only, no training: the A5 checkpoint is taken from that kernel's output and scored on
the 300-record ORD test set, the same one the published `ReactionT5v2-retrosynthesis` checkpoint
was scored on. That fills the one empty cell of the comparison the thesis is built around —
reference checkpoint and this work's model, each on both test sets:

| | ORD (300) | USPTO-50K |
|---|---|---|
| ReactionT5, no fine-tuning | 43.7% exact top-1 | 16.3% |
| A5 (CompoundT5 + USPTO-50K) | *this run* | 50.8% (5003 records) |

The asymmetry in the top row is the contamination argument: ReactionT5 was pretrained on ~1.5M ORD
reactions from the same database the ORD test is drawn from, so 43.7% is an upper bound while
16.3% is honest. This run measures the mirror image — a model that has seen neither ORD reactions
in pretraining nor ORD data in fine-tuning, evaluated on ORD. It is a cross-source generalization
test, and the number is expected to fall well below the USPTO one; the point is to state by how
much, from a model whose training data is fully accounted for.

**Cost:** ~15 min, no training. The checkpoint arrives as this kernel's input via `kernel_sources`,
so nothing is re-uploaded.

**Before running:** Settings -> **Internet** on, **GPU T4 x2** on. Run as **Save & Run All (Commit)**.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
import glob

# A5's output is mounted read-only; the trained weights sit in its `final` folder.
model_dir = next(glob.iglob("/kaggle/input/**/model1_compoundt5_uspto_rsmiles10/final", recursive=True))
print("checkpoint:", model_dir, sorted(os.listdir(model_dir))[:6])

In [ ]:
!python scripts/models/run_reactiont5_topk.py \
    --input "data/v2_ord_eval_targets.json" --t5-model "{model_dir}" \
    --num-beams 10 --device cuda \
    --output "/kaggle/working/A5_rsmiles10_e2_ord300_topk.json"

In [ ]:
import json
data = json.load(open("/kaggle/working/A5_rsmiles10_e2_ord300_topk.json"))
print(json.dumps(data["summary"], indent=2))